# Experiment 1: Dense Baseline - Moondream SFT Fine-Tuning

This notebook fine-tunes Moondream 2B on FloodNet VQA using Supervised Fine-Tuning (Cross-Entropy Loss).

**What you need:**
1. GPU runtime (T4 or better recommended)
2. FloodNet VQA dataset from https://github.com/BinaLab/FloodNet-VQA

**Training happens locally** - requires GPU.

## Step 1: Configuration

Fixed settings for fair comparison across all 4 experiments.

In [ ]:
CONFIG = {
    # Experiment metadata
    "experiment_name": "Experiment 1: Dense Baseline",
    "experiment_id": "dense_baseline",

    # Model settings
    "model_name": "vikhyatk/moondream2",
    "model_revision": "2024-08-26",

    # LoRA settings (FIXED across experiments)
    "lora_rank": 32,
    "lora_alpha": 64,
    "lora_dropout": 0.05,

    # Training settings (FIXED across experiments)
    "epochs": 3,
    "samples_per_epoch": 500,
    "batch_size": 2,
    "learning_rate": 2e-4,
    "weight_decay": 0.01,
    "warmup_steps": 50,
    "max_grad_norm": 1.0,

    # Reproducibility
    "seed": 42,

    # Evaluation
    "max_new_tokens": 32,
}

# Derived values
CONFIG["total_steps"] = CONFIG["epochs"] * (CONFIG["samples_per_epoch"] // CONFIG["batch_size"])

print("Configuration:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

## Step 2: Install Dependencies

In [ ]:
!pip install -q transformers peft accelerate pillow tqdm einops

## Step 3: Upload Your Data

Run this cell and upload your `floodnet.zip` file when prompted.

In [ ]:
from google.colab import files
import os

print("Please upload your floodnet.zip file:")
uploaded = files.upload()

!unzip -q floodnet.zip -d /content/
print("\nExtracted! Contents:")
!ls /content/floodnet/

## Step 4: Import Libraries & Setup

In [ ]:
import json
import re
import random
from pathlib import Path
from collections import defaultdict
from typing import Optional, List, Dict, Any, Tuple

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.amp import autocast, GradScaler

from PIL import Image
from tqdm.auto import tqdm
import numpy as np

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CONFIG["seed"])

# Paths
ROOT = Path("/content/floodnet")
ANN_ROOT = ROOT / "data"
IMG_ROOT = ROOT / "Images"

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Step 5: Helper Functions

In [ ]:
def load_json(path: Path) -> Any:
    """Load JSON file."""
    with open(path, "r") as f:
        return json.load(f)


def normalize_answer(text: str) -> str:
    """
    Normalize text for answer comparison.
    Handles verbose model outputs like 'Yes, there is a flooded building' -> 'yes'
    """
    text = str(text).lower().strip()
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def extract_answer(prediction: str, ground_truth: str) -> str:
    """
    Extract the core answer from verbose model outputs.

    Examples:
        'Yes, there is a flooded building.' -> 'yes' (if gt is 'yes')
        'There are two buildings in the image.' -> '2' (if gt is '2')
        'The area is dry' -> 'non flooded' (synonym matching)
    """
    pred_norm = normalize_answer(prediction)
    gt_norm = normalize_answer(ground_truth)

    SYNONYMS = {
        "non flooded": ["non flooded", "dry", "not flooded", "unflooded"],
        "flooded": ["flooded", "wet", "submerged", "underwater"],
        "destroyed": ["destroyed", "damaged", "wrecked", "collapsed"],
        "yes": ["yes", "yeah", "correct", "true", "affirmative"],
        "no": ["no", "nope", "false", "negative"],
    }

    if pred_norm == gt_norm:
        return pred_norm

    for canonical, synonyms in SYNONYMS.items():
        if gt_norm == canonical or gt_norm in synonyms:
            for syn in synonyms:
                if syn in pred_norm or pred_norm.startswith(syn.split()[0]):
                    return canonical

    if gt_norm in ["yes", "no"]:
        if pred_norm.startswith("yes") or "yes" in pred_norm.split()[:3]:
            return "yes"
        if pred_norm.startswith("no") or "no" in pred_norm.split()[:3]:
            return "no"

    try:
        gt_num = int(gt_norm)
        numbers = re.findall(r'\b(\d+)\b', pred_norm)
        if numbers:
            return numbers[0]
        word_to_num = {
            "zero": "0", "one": "1", "two": "2", "three": "3", "four": "4",
            "five": "5", "six": "6", "seven": "7", "eight": "8", "nine": "9",
            "ten": "10"
        }
        for word, num in word_to_num.items():
            if word in pred_norm:
                return num
    except ValueError:
        pass

    categories = ["high", "moderate", "low", "flooded", "non flooded", "destroyed"]
    if gt_norm in categories:
        for cat in categories:
            if cat in pred_norm:
                return cat

    return pred_norm


def check_answer(prediction: str, ground_truth: str) -> bool:
    """Check if prediction matches ground truth with smart extraction."""
    extracted = extract_answer(prediction, ground_truth)
    gt_norm = normalize_answer(ground_truth)
    return extracted == gt_norm


print("Helper functions defined!")
print("\nTesting answer extraction:")
test_cases = [
    ("Yes, there is a flooded building.", "yes"),
    ("No", "no"),
    ("There are two buildings.", "2"),
    ("The density is high.", "high"),
]
for pred, gt in test_cases:
    result = check_answer(pred, gt)
    print(f"  '{pred[:40]}' vs '{gt}' -> {result}")

## Step 6: Load Model & Apply LoRA

In [ ]:
print(f"Loading {CONFIG['model_name']} (revision: {CONFIG['model_revision']})...")
print("This may take a few minutes on first run.\n")

model = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_name"],
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map={"": device},
    revision=CONFIG["model_revision"]
)

tokenizer = AutoTokenizer.from_pretrained(
    CONFIG["model_name"],
    revision=CONFIG["model_revision"]
)

model.tokenizer = tokenizer

print(f"✓ Model loaded: {type(model).__name__}")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
def setup_lora(model, config: dict):
    """
    Apply LoRA to the model's text decoder.
    Freezes vision encoder and applies LoRA to language model.
    """
    # Freeze vision encoder
    for param in model.vision_encoder.parameters():
        param.requires_grad = False
    print("✓ Froze vision encoder")

    # Configure LoRA
    lora_config = LoraConfig(
        r=config["lora_rank"],
        lora_alpha=config["lora_alpha"],
        lora_dropout=config["lora_dropout"],
        bias="none",
        task_type=TaskType.CAUSAL_LM,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
            "fc1", "fc2", "dense", "query", "key", "value"
        ],
    )

    # Apply LoRA to text model
    peft_model = get_peft_model(model.text_model, lora_config)
    model.text_model = peft_model
    print("✓ Applied LoRA to text model")

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())

    print(f"\n  LoRA rank: {config['lora_rank']}")
    print(f"  Trainable: {trainable:,} ({100 * trainable / total:.2f}%)")
    print(f"  Total: {total:,}")

    return model

model = setup_lora(model, CONFIG)

## Step 7: Load FloodNet Data

In [ ]:
class FloodNetVQADataset(Dataset):
    """
    PyTorch Dataset for FloodNet VQA.
    """
    def __init__(self, annotations: List[dict], img_dir: Path, max_samples: int = None):
        self.img_dir = Path(img_dir)
        self.samples = []

        annotations = annotations[:max_samples] if max_samples else annotations

        for ann in annotations:
            img_path = self.img_dir / ann["Image_ID"]
            if img_path.exists():
                self.samples.append({
                    "img_path": img_path,
                    "question": ann["Question"],
                    "answer": str(ann["Ground_Truth"]),
                    "type": ann.get("Question_Type", "unknown"),
                    "image_id": ann["Image_ID"]
                })

        print(f"  Loaded {len(self.samples)} samples from {img_dir.name}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        image = Image.open(sample["img_path"]).convert("RGB")
        return {
            "image": image,
            "question": sample["question"],
            "answer": sample["answer"],
            "type": sample["type"],
            "image_id": sample["image_id"]
        }


# Load data
print("Loading FloodNet annotations...")
train_ann = load_json(ANN_ROOT / "train_annotations.json")
val_ann = load_json(ANN_ROOT / "valid_annotations.json")
test_ann = load_json(ANN_ROOT / "test_annotations.json")
print(f"Annotations: Train={len(train_ann)} | Val={len(val_ann)} | Test={len(test_ann)}")

print("\nCreating datasets...")
train_dataset = FloodNetVQADataset(train_ann, IMG_ROOT / "train_images")
val_dataset = FloodNetVQADataset(val_ann, IMG_ROOT / "valid_images")
test_dataset = FloodNetVQADataset(test_ann, IMG_ROOT / "test_images")

## Step 8: Trainer Class (Cross-Entropy Loss)

In [ ]:
class MoondreamTrainer:
    """
    Trainer for Moondream SFT with cross-entropy loss.
    """
    def __init__(self, model, tokenizer, config: dict):
        self.model = model
        self.tokenizer = tokenizer
        self.config = config
        self.device = next(model.parameters()).device

        trainable_params = [p for p in model.parameters() if p.requires_grad]
        self.optimizer = AdamW(
            trainable_params,
            lr=config["learning_rate"],
            weight_decay=config["weight_decay"]
        )

        self.scaler = GradScaler("cuda")

        self.current_step = 0

        print(f"✓ Trainer initialized (lr={config['learning_rate']})")

    def _get_lr(self) -> float:
        """Linear warmup then constant."""
        if self.current_step < self.config["warmup_steps"]:
            return self.config["learning_rate"] * (self.current_step + 1) / self.config["warmup_steps"]
        return self.config["learning_rate"]

    def _update_lr(self):
        lr = self._get_lr()
        for pg in self.optimizer.param_groups:
            pg["lr"] = lr

    def compute_loss(self, images, questions, answers) -> torch.Tensor:
        """Compute cross-entropy loss for a batch."""
        total_loss = 0.0
        valid = 0

        for image, question, answer in zip(images, questions, answers):
            try:
                with torch.no_grad():
                    image_embeds = self.model.encode_image(image)

                prompt = f"\n\nQuestion: {question}\n\nAnswer:"
                full_text = f"{prompt} {answer}"

                prompt_ids = self.tokenizer.encode(prompt, add_special_tokens=False)
                full_ids = self.tokenizer.encode(full_text, add_special_tokens=False)

                input_ids = torch.tensor([full_ids], device=self.device)

                labels = input_ids.clone()
                labels[0, :len(prompt_ids)] = -100

                text_embeds = self.model.text_model.get_input_embeddings()(input_ids)
                inputs_embeds = torch.cat([image_embeds, text_embeds], dim=1)

                image_labels = torch.full(
                    (1, image_embeds.shape[1]), -100, dtype=torch.long, device=self.device
                )
                labels = torch.cat([image_labels, labels], dim=1)

                outputs = self.model.text_model(
                    inputs_embeds=inputs_embeds,
                    labels=labels,
                    return_dict=True
                )

                total_loss += outputs.loss
                valid += 1

            except Exception as e:
                print(f"Error: {e}")
                continue

        if valid == 0:
            return torch.tensor(0.0, device=self.device, requires_grad=True)
        return total_loss / valid

    def train_step(self, batch: dict) -> float:
        """Single training step."""
        self.model.train()
        self._update_lr()

        with autocast("cuda", dtype=torch.float16):
            loss = self.compute_loss(
                batch["image"], batch["question"], batch["answer"]
            )

        self.optimizer.zero_grad()
        self.scaler.scale(loss).backward()

        self.scaler.unscale_(self.optimizer)
        torch.nn.utils.clip_grad_norm_(
            [p for p in self.model.parameters() if p.requires_grad],
            max_norm=self.config["max_grad_norm"]
        )

        self.scaler.step(self.optimizer)
        self.scaler.update()

        self.current_step += 1
        return loss.item()

    @torch.no_grad()
    def generate(self, image, question: str, max_tokens: int = 32) -> str:
        """Generate answer for a single image-question pair."""
        self.model.eval()

        image_embeds = self.model.encode_image(image)

        prompt = f"\n\nQuestion: {question}\n\nAnswer:"
        input_ids = self.tokenizer.encode(prompt, add_special_tokens=False)

        embed_layer = self.model.text_model.base_model.model.transformer.embd.wte

        generated = []

        for _ in range(max_tokens):
            all_ids = input_ids + generated
            ids_tensor = torch.tensor([all_ids], device=self.device)
            text_embeds = embed_layer(ids_tensor)
            inputs_embeds = torch.cat([image_embeds, text_embeds], dim=1)

            outputs = self.model.text_model(
                inputs_embeds=inputs_embeds,
                use_cache=False,
                return_dict=True
            )

            next_token = torch.argmax(outputs.logits[0, -1, :]).item()

            if next_token == self.tokenizer.eos_token_id:
                break

            generated.append(next_token)

            if len(generated) > 1 and "\n" in self.tokenizer.decode([next_token]):
                break

        return self.tokenizer.decode(generated, skip_special_tokens=True).strip()


trainer = MoondreamTrainer(model, tokenizer, CONFIG)

## Step 9: Evaluation Function

In [ ]:
def evaluate(
    trainer: MoondreamTrainer,
    dataset: FloodNetVQADataset,
    name: str = "Evaluation",
    max_samples: int = None
) -> Tuple[dict, float]:
    """
    Evaluate model on dataset.

    Returns:
        results: dict with predictions and per-type metrics
        accuracy: overall accuracy
    """
    print(f"\n{'='*60}")
    print(f"EVALUATING: {name}")
    print(f"{'='*60}")

    n_samples = min(len(dataset), max_samples) if max_samples else len(dataset)
    indices = list(range(n_samples))

    predictions = []
    by_type = defaultdict(lambda: {"correct": 0, "total": 0})

    for idx in tqdm(indices, desc=f"Evaluating {name}"):
        sample = dataset[idx]

        pred = trainer.generate(
            sample["image"],
            sample["question"],
            max_tokens=CONFIG["max_new_tokens"]
        )

        correct = check_answer(pred, sample["answer"])

        predictions.append({
            "question": sample["question"],
            "predicted": pred,
            "ground_truth": sample["answer"],
            "correct": correct
        })

        by_type[sample["type"]]["total"] += 1
        if correct:
            by_type[sample["type"]]["correct"] += 1

    total_correct = sum(1 for p in predictions if p["correct"])
    accuracy = total_correct / len(predictions)

    print(f"\n{name} Results:")
    print(f"  Overall Accuracy: {accuracy:.4f} ({total_correct}/{len(predictions)})")
    print(f"\n  By Question Type:")
    for qtype in sorted(by_type.keys()):
        m = by_type[qtype]
        acc = m["correct"] / m["total"] if m["total"] > 0 else 0
        print(f"    {qtype}: {acc:.4f} ({m['correct']}/{m['total']})")

    return {
        "predictions": predictions,
        "by_type": dict(by_type)
    }, accuracy

print("Evaluation function defined!")

## Step 10: Baseline Evaluation

In [ ]:
baseline_results, baseline_acc = evaluate(
    trainer,
    test_dataset,
    name="Baseline (Pre-training)"
)

print(f"\n📊 Baseline accuracy: {baseline_acc:.4f}")

## Step 11: Training Function

In [ ]:
def collate_fn(batch):
    """Keep images as PIL for processing."""
    return {
        "image": [item["image"] for item in batch],
        "question": [item["question"] for item in batch],
        "answer": [item["answer"] for item in batch],
        "type": [item["type"] for item in batch],
        "image_id": [item["image_id"] for item in batch]
    }


def train(trainer: MoondreamTrainer, train_dataset: Dataset, config: dict) -> List[dict]:
    """Training loop."""
    print(f"\n{'='*60}")
    print("STARTING TRAINING")
    print(f"{'='*60}")
    print(f"Epochs: {config['epochs']}")
    print(f"Batch size: {config['batch_size']}")
    print(f"Samples/epoch: {config['samples_per_epoch']}")
    print(f"Total steps: ~{config['total_steps']}")

    history = []

    for epoch in range(1, config["epochs"] + 1):
        print(f"\n{'='*60}")
        print(f"EPOCH {epoch}/{config['epochs']}")
        print(f"{'='*60}")

        # Subsample
        if config["samples_per_epoch"] < len(train_dataset):
            indices = random.sample(range(len(train_dataset)), config["samples_per_epoch"])
            epoch_dataset = torch.utils.data.Subset(train_dataset, indices)
        else:
            epoch_dataset = train_dataset

        dataloader = DataLoader(
            epoch_dataset,
            batch_size=config["batch_size"],
            shuffle=True,
            collate_fn=collate_fn,
            num_workers=0
        )

        epoch_losses = []
        pbar = tqdm(dataloader, desc=f"Epoch {epoch}")

        for batch in pbar:
            loss = trainer.train_step(batch)
            epoch_losses.append(loss)

            avg_loss = np.mean(epoch_losses[-50:])
            pbar.set_postfix({
                "loss": f"{loss:.4f}",
                "avg": f"{avg_loss:.4f}",
                "step": trainer.current_step
            })

        avg_loss = np.mean(epoch_losses)
        print(f"\nEpoch {epoch}: avg_loss={avg_loss:.4f}, steps={trainer.current_step}")

        history.append({
            "epoch": epoch,
            "avg_loss": float(avg_loss),
            "steps": trainer.current_step
        })

        torch.cuda.empty_cache()

    print(f"\n{'='*60}")
    print(f"TRAINING COMPLETE (step {trainer.current_step})")
    print(f"{'='*60}")

    return history

print("Training function defined!")

## Step 12: Train the Model

In [ ]:
history = train(trainer, train_dataset, CONFIG)

## Step 13: Final Evaluation

In [ ]:
final_results, final_acc = evaluate(
    trainer,
    test_dataset,
    name="After Training"
)

print(f"\n{'='*60}")
print("COMPARISON")
print(f"{'='*60}")
print(f"Baseline accuracy: {baseline_acc:.4f}")
print(f"Final accuracy:    {final_acc:.4f}")
print(f"Improvement:       {final_acc - baseline_acc:+.4f}")

## Step 14: Save & Download Results

In [ ]:
results = {
    "experiment": CONFIG["experiment_name"],
    "experiment_id": CONFIG["experiment_id"],
    "model": f"Moondream 2B + LoRA (rank={CONFIG['lora_rank']})",
    "training_method": "Supervised Fine-Tuning (Cross-Entropy)",

    "config": CONFIG,

    "training": {
        "epochs": CONFIG["epochs"],
        "samples_per_epoch": CONFIG["samples_per_epoch"],
        "batch_size": CONFIG["batch_size"],
        "learning_rate": CONFIG["learning_rate"],
        "final_step": trainer.current_step,
        "history": history
    },

    "results": {
        "baseline_accuracy": float(baseline_acc),
        "final_accuracy": float(final_acc),
        "improvement": float(final_acc - baseline_acc)
    },

    "by_question_type": {
        qt: {
            "accuracy": m["correct"] / m["total"] if m["total"] > 0 else 0,
            "correct": m["correct"],
            "total": m["total"]
        }
        for qt, m in final_results["by_type"].items()
    },

    "sample_predictions": final_results["predictions"][:20]
}

output_path = f"/content/{CONFIG['experiment_id']}_results.json"
with open(output_path, "w") as f:
    json.dump(results, f, indent=2)

print(f"✓ Results saved to {output_path}")
print(f"\nFinal accuracy: {final_acc:.4f}")

In [ ]:
lora_path = f"/content/moondream_floodnet_{CONFIG['experiment_id']}"
model.text_model.save_pretrained(lora_path)
print(f"✓ LoRA adapter saved to {lora_path}")

In [ ]:
from google.colab import files

files.download(output_path)

!zip -r {lora_path}.zip {lora_path}
files.download(f"{lora_path}.zip")